# Communication

> Agents that communicate: comm-core's networks, with the agents of an environment as nodes.

In [ ]:
#| default_exp comm.core

In [ ]:
#| hide
from nbdev.showdoc import *

Multi-agent methods that need agents to talk to each other model the communication with
[comm-core](https://github.com/Ahmed-Khaled-Saleh/comm-core), a separate library that knows nothing about
MARL, world models or federated learning: topologies (who can talk to whom), protocols (when a message is
transmitted), channels and backends (what happens to it: perfect, AWGN, later Sionna, ns-3, ...) and metrics.
stable-marl only connects it to its agents; the communication itself is comm-core's.

`AgentNetwork` is a comm-core `Network` whose nodes are the agents of one environment (node ids
`agent_0 .. agent_{n-1}`, see `node_id`). The learning code decides *what* is sent and *to whom*; the network
decides what arrives. Its `exchange` is one round of communication: every agent with a payload sends it to each of
its neighbours in the topology, the network steps once, and it returns what each agent received, as given by the
channel (`TransmissionResult.payload`, so channels that corrupt payloads are taken into account).

comm-core is an optional dependency (`pip install 'stable-marl[comm]'`), imported only when an `AgentNetwork`
is created. Each network has its own random generator (seeded by `reset`), which the channels draw from instead
of Python's global one, so runs are reproducible and the global generator is left as it was.

In [ ]:
#| export
from __future__ import annotations

import contextlib
import random
from types import SimpleNamespace
from typing import Any, Callable

import numpy as np

In [ ]:
#| export
def _comm_core() -> SimpleNamespace:
    "comm-core's building blocks, imported on first use."
    try:
        from comm_core import core, topology, channel, protocol, backend, network
    except ImportError as e:
        if getattr(e, 'name', None) == 'comm_core':
            raise ModuleNotFoundError("stable_marl.comm needs comm-core: pip install 'stable-marl[comm]'") from e
        raise ImportError(f"stable_marl.comm needs a comm-core with networks and backends ({e})") from e
    return SimpleNamespace(Node=core.Node, Message=core.Message, Topology=topology.Topology,
                           GraphTopology=topology.GraphTopology, Link=topology.Link,
                           FullMeshTopology=topology.FullMeshTopology, StarTopology=topology.StarTopology,
                           IdentityChannel=channel.IdentityChannel, Protocol=protocol.Protocol,
                           IdealProtocol=protocol.IdealProtocol, AnalyticalBackend=backend.AnalyticalBackend,
                           Network=network.Network)


def node_id(agent: int) -> str:
    "The comm-core node id of agent `agent`: 'agent_<agent>'."
    return f'agent_{int(agent)}'


def agent_index(node: str) -> int:
    "The agent index of a comm-core node id ('agent_3' -> 3)."
    return int(str(node).rsplit('_', 1)[1])


def make_topology(
    spec, # 'full_mesh', 'star' (around agent `center`), 'none' (no links), or a comm-core `Topology`
    num_agents: int, # Number of agents (nodes `agent_0 ..`)
    center: int = 0 # Center of the 'star' topology
):
    "A comm-core topology over the agents."
    cc = _comm_core()
    ids = [node_id(a) for a in range(num_agents)]
    if isinstance(spec, cc.Topology):
        return spec
    if spec == 'full_mesh':
        return cc.FullMeshTopology(ids)
    if spec == 'star':
        return cc.StarTopology(node_id(center), ids)
    if spec == 'none':
        return cc.GraphTopology([])
    raise ValueError(f"topology: 'full_mesh', 'star', 'none' or a comm-core Topology, not {spec!r}")


def message_bits(
    payload, # What is sent (anything numpy can turn into an array)
    bits_per_element: int | None = None # Bits per number (e.g. after quantization); default: the array's own dtype
) -> int: # Size in bits
    "Size of a message, for the channel and the metrics."
    arr = np.asarray(payload)
    return int(arr.size * bits_per_element) if bits_per_element else int(arr.nbytes * 8)


@contextlib.contextmanager
def _rng_scope(rng: random.Random):
    "Python's global generator replaced by `rng` (comm-core's channels draw from the global one)."
    outer = random.getstate()
    random.setstate(rng.getstate())
    try:
        yield
    finally:
        rng.setstate(random.getstate())
        random.setstate(outer)

In [ ]:
#| export
class AgentNetwork:
    "A comm-core `Network` whose nodes are the agents of one environment."
    def __init__(
        self,
        num_agents: int, # Number of agents (nodes `agent_0 ..`)
        topology='full_mesh', # 'full_mesh', 'star', 'none' or a comm-core `Topology`
        channel=None, # A comm-core `Channel` (default: a perfect channel), used through an `AnalyticalBackend`
        backend=None, # A comm-core `CommunicationBackend` (instead of `channel`)
        protocol=None, # A comm-core `Protocol` class / factory, instantiated at every reset (default `IdealProtocol`)
        center: int = 0, # Center of the 'star' topology
        bits_per_element: int | None = None, # Message size accounting (see `message_bits`)
        seed: int | None = None # Seed of the channel's random generator
    ):
        cc = _comm_core()
        if channel is not None and backend is not None:
            raise ValueError("pass a channel or a backend, not both")
        self.num_agents, self.center, self.bits_per_element = int(num_agents), center, bits_per_element
        self.backend = backend if backend is not None else cc.AnalyticalBackend(channel if channel is not None else cc.IdentityChannel())
        self._protocol = protocol if protocol is not None else cc.IdealProtocol
        self.topology = make_topology(topology, self.num_agents, center)
        self.rng = random.Random(seed)
        self.last_results: list = []
        self.reset(seed)

    def reset(self, seed: int | None = None):
        "A new network (empty queues and inboxes, fresh metrics) for a new episode; `seed` reseeds the channel."
        cc = _comm_core()
        if seed is not None:
            self.rng.seed(seed)
        self.backend.reset()
        protocol = self._protocol() if callable(self._protocol) and not isinstance(self._protocol, cc.Protocol) else self._protocol
        self.network = cc.Network(nodes=[cc.Node(node_id(a)) for a in range(self.num_agents)], topology=self.topology,
                                  protocol=protocol, backend=self.backend)
        self.last_results = []

    @property
    def metrics(self):
        "comm-core's metrics of the episode so far (transmissions, losses, bits, latency, energy)."
        return self.network.metrics.summary()

    def set_topology(self, topology):
        "Change who can talk to whom (e.g. agents that moved out of range)."
        self.topology = make_topology(topology, self.num_agents, self.center)
        self.network.topology = self.topology

    def neighbors(self, agent: int) -> list[int]:
        "The agents `agent` can send to."
        return sorted(agent_index(n) for n in self.topology.neighbors(node_id(agent)))

    def send(
        self,
        sender: int, # Agent sending
        receiver: int, # Agent it is sent to (must be a neighbour)
        payload: Any, # What is sent
        type: str = 'message', # comm-core's message type (e.g. 'latent', 'observation')
        size_bits: int | None = None, # Default: `message_bits(payload)`
        **metadata
    ):
        "Queue a message; it is transmitted at the next `step`."
        cc = _comm_core()
        size = message_bits(payload, self.bits_per_element) if size_bits is None else int(size_bits)
        self.network.send(cc.Message(sender=node_id(sender), receiver=node_id(receiver), payload=payload, type=type,
                                     timestamp=self.network.time, size_bits=size, metadata=metadata))

    def step(self, dt: float = 1.0) -> list:
        "Transmit what the protocol schedules; the comm-core `TransmissionResult`s."
        with _rng_scope(self.rng):
            results = self.network.step(dt)
        for a in range(self.num_agents):            # delivered payloads are read from the results
            self.network.receive(node_id(a))
        self.last_results = results
        return results

    def exchange(
        self,
        payloads: dict[int, Any], # Each agent's payload (None: the agent sends nothing)
        type: str = 'message', # comm-core's message type
        dt: float = 1.0 # Time step of the network
    ) -> dict[int, dict[int, Any]]: # received[i][j]: what agent i received from agent j
        "One round: every agent sends its payload to each of its neighbours, then one network step."
        for a, payload in payloads.items():
            if payload is None:
                continue
            for b in self.neighbors(a):
                self.send(a, b, payload, type=type)
        received: dict[int, dict[int, Any]] = {a: {} for a in range(self.num_agents)}
        for r in self.step(dt):
            if r.success:
                received[agent_index(r.transmission.receiver)][agent_index(r.transmission.sender)] = r.payload
        return received

In [ ]:
show_doc(AgentNetwork.reset)

In [ ]:
show_doc(AgentNetwork.set_topology)

In [ ]:
show_doc(AgentNetwork.neighbors)

In [ ]:
show_doc(AgentNetwork.send)

In [ ]:
show_doc(AgentNetwork.step)

In [ ]:
show_doc(AgentNetwork.exchange)

### Tests

In [ ]:
# imports on their own (the docs build runs the cells with imports; the tests need comm-core)
from fastcore.test import test_eq, test_fail

In [ ]:
from comm_core.channel import AWGNChannel
net = AgentNetwork(3)
latents = {a: np.full(4, a, np.float32) for a in range(3)}
received = net.exchange(latents, type='latent')
test_eq(sorted(received[0]), [1, 2])                                      # full mesh: everyone hears everyone
assert np.array_equal(received[2][1], latents[1])
m = net.metrics
test_eq((m.transmissions, m.successful, m.bits_sent), (6, 6, 6 * 4 * 32))
# star: the leaves only talk to the center
net = AgentNetwork(3, topology='star', center=0)
test_eq((net.neighbors(0), net.neighbors(1)), ([1, 2], [0]))
received = net.exchange(latents)
test_eq((sorted(received[0]), sorted(received[1])), ([1, 2], [0]))
# a lossy channel: reproducible with a seed, Python's global generator untouched
def losses(seed):
    net = AgentNetwork(4, channel=AWGNChannel(snr_db=5, packet_error_rate=0.5), seed=seed)
    return [sorted(net.exchange(latents | {3: latents[0]})[0]) for _ in range(5)], net.metrics
random.seed(1); before = random.random()
first, m = losses(0)
test_eq(losses(0)[0], first)
assert 0 < m.failed < m.transmissions and m.transmissions == 5 * 12
random.seed(1); test_eq(random.random(), before)
# reset: fresh metrics; quantized size accounting; argument checks
net = AgentNetwork(2, bits_per_element=8)
net.exchange({0: np.zeros(10), 1: None})
test_eq(net.metrics.bits_sent, 80)
net.reset()
test_eq(net.metrics.transmissions, 0)
test_fail(lambda: AgentNetwork(2, topology='ring'), contains='full_mesh')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()